# Exploratory correlations and figures

**Purpose:** Run microbiome-metabolome, microbiome-cognition, diet-microbiome, and diet-metabolome association analyses and generate manuscript figures.

**Expected inputs**
- `../data/Nightingale.csv`
- `../data/Metabolon.csv`
- `../data/UCSD Known.csv`
- `../data/plasma_metadata_matched_main_outcomes.tsv`

**Main outputs**
- `../output_files/*.csv`
- `../figures/*.png`

> Notes for reuse: data files are not included in this repository. Update paths in the cells below to match the local location of the approved, de-identified data release. Notebook outputs have been cleared for public sharing.


In [ ]:
from pathlib import Path

PROJECT_ROOT = Path('..').resolve()
DATA_DIR = PROJECT_ROOT / 'data'
OUTPUT_DIR = PROJECT_ROOT / 'output_files'
FIGURE_DIR = PROJECT_ROOT / 'figures'

for directory in [DATA_DIR, OUTPUT_DIR, FIGURE_DIR]:
    directory.mkdir(parents=True, exist_ok=True)


In [ ]:
# Imports
import pandas as pd
from biom import load_table
import numpy as np
import qiime2 as q2
import patsy
import statsmodels.api as sm
from skbio.stats.composition import clr, multiplicative_replacement
from scipy.stats import spearmanr, pearsonr, kruskal, mannwhitneyu
from statsmodels.stats.multitest import multipletests
from sklearn.linear_model import LinearRegression

import re
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.patches import Patch
import pingouin as pg
import networkx as nx


In [ ]:
# Load Feature Tables

fts = {
    'Nightingale': pd.read_csv('../data/Nightingale.csv', dtype={'sample_name': str}).set_index('sample_name'),
       'Metabolon': pd.read_csv('../data/Metabolon.csv', dtype={'sample_name': str}).set_index('sample_name'),
       'UCSD': pd.read_csv('../data/UCSD Known.csv', dtype={'sample_name': str}).set_index('sample_name'),
       'Metagenomics': load_table('../data/micov_filtered_feature-table_matched.biom').to_dataframe().T
      }


In [ ]:
# And metadata
md = pd.read_csv('../data/plasma_metadata_matched_main_outcomes.tsv', sep = '\t', dtype={'UCSD': str, 'Unnamed: 0': str, 'sample_name': str}).set_index('sample_name')


In [ ]:
# Save non clr version of feature table
non_clr = fts['Metagenomics']


In [ ]:
# Multiplicative CLR for metagenomics 

def multiplicative_clr(df):
    metagenomic_counts = np.array(df)  

    # Replace zeros using multiplicative replacement
    metagenomic_counts = multiplicative_replacement(metagenomic_counts)

    # Apply CLR transformation
    clr_transformed_data = clr(metagenomic_counts)

    df = pd.DataFrame(clr_transformed_data,index=df.index, columns=df.columns)
    return df


fts['Metagenomics'] = multiplicative_clr(fts['Metagenomics'])


## Correlate all microbes with all metabolites across 7 platforms while controlling for confounders


In [ ]:
resid = {}

# covariates
covars_raw = md[["NACCAGE", 'SEX', "site"]].copy()

# make sure numeric cols are numeric (coerce weird strings to NaN)
for c in ["NACCAGE"]:
    covars_raw[c] = pd.to_numeric(covars_raw[c], errors="coerce")

# one-hot encode categoricals
covars = pd.get_dummies(covars_raw, columns=["site",'SEX'], drop_first=True)

# replace inf -> NaN, then drop rows with any missing covariate
covars = covars.replace([np.inf, -np.inf], np.nan).dropna(axis=0)


def residualize(matrix: pd.DataFrame, covars: pd.DataFrame) -> pd.DataFrame:
    # align samples
    common = matrix.index.intersection(covars.index)
    X_all = covars.loc[common]

    out = pd.DataFrame(index=common, columns=matrix.columns, dtype=float)

    for col in matrix.columns:
        y_all = pd.to_numeric(matrix.loc[common, col], errors="coerce")
        y_all = y_all.replace([np.inf, -np.inf], np.nan)

        ok = y_all.notna() & np.isfinite(y_all) & X_all.notna().all(axis=1)
        if ok.sum() < (X_all.shape[1] + 2):
            continue  # not enough rows to fit

        X = X_all.loc[ok]
        y = y_all.loc[ok]

        model = LinearRegression().fit(X, y)
        out.loc[ok, col] = y - model.predict(X)

    return out

for ft, mat in fts.items():
    resid[ft] = residualize(mat, covars)


In [ ]:
#correlating all microbes + metabolites 
results_df = {}


for ft in resid.keys():
    if ft != 'Metagenomics':
        print(ft)
        # 1) Align samples and drop any row with NaNs for speed
        X = resid[ft].sort_index()
        X = X.dropna(axis=0, how='any')
        Y = resid['Metagenomics'].sort_index()
        Y = Y.loc[Y.index.intersection(X.index)]
        XY = (
            pd.concat([X, Y], axis=1, join="inner")
        )
        m, n = X.shape[1], Y.shape[1]
        Xc = XY.iloc[:, :m].to_numpy()
        Yc = XY.iloc[:, m:].to_numpy()

        # 2) One-shot Spearman on all columns
        corr_full, p_full = spearmanr(np.concatenate([Xc, Yc], axis=1), axis=0)
        # 3) Take the metabolite×microbe block
        corr = corr_full[:m, m:]
        pvals = p_full[:m, m:]

        # 4) FDR
        qvals = multipletests(pvals.ravel(), method="fdr_bh")[1].reshape(pvals.shape)

        # 5) Wrap as DataFrames (optional)
        corr_df = pd.DataFrame(corr, index=X.columns, columns=Y.columns)
        p_df    = pd.DataFrame(pvals, index=X.columns, columns=Y.columns)
        q_df    = pd.DataFrame(qvals, index=X.columns, columns=Y.columns)


        # Long-form results: one row per (metabolite, microbe)
        results_df[ft] = (
            corr_df.stack().rename("corr").to_frame()
            .join(p_df.stack().rename("p_value"))
            .join(q_df.stack().rename("q_value"))
            .reset_index()
            .rename(columns={"level_0": "metabolite", "level_1": "microbe"})
        )
        
        results_df[ft]['Platform'] = ft


In [ ]:
# Save signifianct results

sig_results = {}

for ft in fts.keys():
    if ft != 'Metagenomics':
        sig_results[ft] = results_df[ft][results_df[ft]['q_value'] < .05]
        
all_sig_results = pd.concat(sig_results.values())
all_results = pd.concat(results_df.values())


In [ ]:
# Short values bu correlation
all_sig_results = all_sig_results.sort_values(by='corr', key=abs, ascending=False)


In [ ]:
def get_most_specific_taxon(taxonomy):
    levels = [t.strip() for t in taxonomy.split(';')]
    
    # Go from most specific → least specific
    for level in reversed(levels):
        # Skip empty or unassigned levels (like s__ or g__)
        if level and not level.endswith('__'):
            return level
    
    return 'Unclassified'


In [ ]:
# Add taxnony to results tables

tax = pd.read_csv('../data/taxonomy.tsv', sep = '\t', index_col='Feature ID')
all_results = all_results.set_index('microbe')
all_sig_results = all_sig_results.set_index('microbe')

all_sig_results['Taxonomy'] = tax['Taxon']
all_sig_results = all_sig_results.reset_index(drop=False)

all_results['Taxonomy'] = tax['Taxon']
all_results = all_results.reset_index(drop=False)


In [ ]:
all_sig_results['species'] = all_sig_results['Taxonomy'].apply(get_most_specific_taxon)


In [ ]:
all_sig_results['feature_id'] = all_sig_results['species'] + '; ' + all_sig_results['microbe']


In [ ]:
all_sig_results['class'] = all_sig_results['Taxonomy'].str.extract(r'c__([^;]+)')


In [ ]:
# Keep top 50 most correlated species
top_microbes = all_sig_results['feature_id'].value_counts().sort_values(ascending=False).head(50)


In [ ]:
all_sig_results = all_sig_results.set_index('feature_id')


In [ ]:
# Create DataFrame for plotting
top_df = top_microbes.reset_index()


In [ ]:
top_df = top_df.set_index('index')


In [ ]:
no_dup = all_sig_results[~all_sig_results.index.duplicated()]


In [ ]:
top_df['class'] = no_dup['class']


In [ ]:
# Create DataFrame for plotting
top_df = top_df.reset_index()

# Plot
plt.figure(figsize=(12, 10))
sns.barplot(
    data=top_df,
    y='index',
    x='feature_id',
    hue='class',
    dodge=False,
    palette='tab20'
)

plt.xlabel("Number of Correlations")
plt.ylabel("Microbial Feature")
plt.title(f"")
plt.legend(title='Class', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.savefig(f'../figures/top_50_correlated_microbes.png', bbox_inches='tight')
plt.show()


## Correlate all microbes + metabolites with MOCA scores


In [ ]:
all_sig_results = all_sig_results.reset_index()


In [ ]:
# Extract OGUs
top_ogus = all_sig_results['microbe'].value_counts().sort_values(ascending=False).head(50).index


In [ ]:
sig_resid = {}

sig_resid['Metagenomics'] = resid['Metagenomics'][top_ogus]


In [ ]:
# Filter feature table to sig features

for ft in resid.keys():
    if ft != 'Metagenomics':
        sig_resid[ft] = resid[ft][resid[ft].columns[resid[ft].columns.isin(all_sig_results['metabolite'])]]


In [ ]:
cog = md['MOCA'].dropna()


In [ ]:
covars_cog = covars.loc[covars.index.intersection(cog.index)]


In [ ]:
# cog = cog.loc[cog.index.intersection(covars_cog.index)]


In [ ]:
common_idx = covars_cog.index.intersection(cog.index)

X = covars_cog.loc[common_idx]
y = cog.loc[common_idx]

cog_resid = y - LinearRegression().fit(X, y).predict(X)


In [ ]:
cog_resid = cog - LinearRegression().fit(covars_cog, cog).predict(covars_cog)


In [ ]:
sig_resid['Metagenomics'] = sig_resid['Metagenomics'].loc[:, ~sig_resid['Metagenomics'].columns.duplicated()]


In [ ]:
#linking all features to MoCA scores

moca_df = {}

for ft in fts.keys():
    print(ft)
    df = sig_resid[ft].copy()
    df['MOCA_corr'] = cog_resid

    df = df.dropna()
    metabs = df.columns

    # Run Spearman for each metabolite vs moca
    r_vals = []
    p_vals = []
    for col in metabs:
        r, p = spearmanr(df[col], df["MOCA_corr"])
        r_vals.append(r)
        p_vals.append(p)

    # Adjust for multiple testing
    q_vals = multipletests(p_vals, method="fdr_bh")[1]

    # Put in a DataFrame
    moca_df[ft] = pd.DataFrame({
        'Platform': ft,
        'Feature': metabs,
        "moca_r": r_vals,
        "moca_p": p_vals,
        "moca_q": q_vals
    })


In [ ]:
# Filter to metagenomics significantly correlated to moca
moca_metag = moca_df['Metagenomics'][moca_df['Metagenomics']['moca_q'] < .05]


In [ ]:
moca_metag = moca_metag.set_index('Feature')
moca_metag['Taxonomy'] = tax['Taxon']


In [ ]:
moca_metag['class'] = moca_metag['Taxonomy'].str.extract(r'c__([^;]+)')


In [ ]:
moca_metag = moca_metag.dropna()


In [ ]:
moca_metag['species'] = moca_metag['Taxonomy'].apply(get_most_specific_taxon)


In [ ]:
moca_metag = moca_metag.reset_index(drop=False)


In [ ]:
moca_metag['feature_id'] = moca_metag['species'] + '; ' + moca_metag['Feature']


In [ ]:
moca_metag = moca_metag[['feature_id', 'class', 'moca_r']]


In [ ]:
moca_metag = moca_metag.sort_values(by='moca_r', ascending=False)


In [ ]:
# Plot top OGUs

# Step 4: Plot
plt.figure(figsize=(12, 9))
sns.barplot(
    data=moca_metag,
    y='moca_r',
    x='feature_id',
    hue='class',
    dodge=False,
    palette='tab20'
)

plt.xlabel("Microbial Feature", fontsize=16)
plt.xticks(rotation=90, fontsize=14)
plt.yticks(fontsize=14)
plt.ylabel("Spearman r", fontsize=16)
plt.title(f"")
plt.legend(title='Class', bbox_to_anchor=(1, 1), loc='upper left')
plt.tight_layout()
plt.savefig(f'../figures/top_moca_correlated_microbes.png', bbox_inches='tight')
plt.show()


In [ ]:
moca_df_sig = moca_df['Metagenomics'][moca_df['Metagenomics']['moca_q'] < .05]


In [ ]:
sig_resid['Metagenomics'][moca_df_sig['Feature'][:-1]].to_csv('../output_files/significant_moca_microbes_residualized.csv')


In [ ]:
# Filter microbe/metabolite table to only include microbes correlated to moca

top_sig = all_sig_results[all_sig_results['feature_id'].isin(moca_metag['feature_id'])]


In [ ]:
# Add all moca correlation dataframes together

moca_df = pd.concat(moca_df.values())

moca_df = moca_df.rename(columns={'Feature': 'metabolite'})


In [ ]:
# Add a significance flag (q < 0.05)
moca_df["metab_sig"] = moca_df["moca_q"] < 0.05

# Merge onto correlation df
top_sig = top_sig.merge(
    moca_df[["metabolite", "metab_sig"]],
    on="metabolite",
    how="left"
)


In [ ]:
# Filter to sig results only
top_sig = top_sig[top_sig['metab_sig'] == True]


In [ ]:
# Only keep the metabolite and moca info we need
moca_keep = moca_df[['metabolite', 'moca_r']]

# Merge onto df
df_merged = top_sig.merge(moca_keep, on='metabolite', how='left')


In [ ]:
# Clean moca metag dataframe and add it to df_merged

moca_metag = moca_metag.rename(columns={'moca_r': 'metag_moca_r'})
df_merged = df_merged.merge(moca_metag, on='feature_id', how='left')


In [ ]:
# Rename columns for clarity
df_merged = df_merged.rename(columns={'corr': 'microbe_metabolite_r', 'moca_r': 'metab_moca_r', 'featureid_clean': 'microbe_w_tax'})


In [ ]:
# Filter columns and save
df_merged = df_merged[['feature_id', 'metabolite', 'microbe_metabolite_r', 'p_value', 'q_value', 'Platform', 'metag_moca_r', 'metab_moca_r']]


In [ ]:
# Sort by microbe-metabolite corr 
df_merged = df_merged.sort_values(by='microbe_metabolite_r', key=abs, ascending=False)


In [ ]:
# Add positive/negative column for plotting

df_merged['metag_moca_dir'] = np.where(df_merged['metag_moca_r'] > 0, 'Positive', 'Negative')
df_merged['metab_moca_dir'] = np.where(df_merged['metab_moca_r'] > 0, 'Positive', 'Negative')


In [ ]:
# save cleaned results dataframe
df_merged.to_csv('../output_files/sig_metab_metag_results.csv')


# Plot all microbe-metabolite correlations associated with MOCA scores


In [ ]:
df_plot = pd.read_csv('../data/Lora_microbe_metabolite_results_NK.csv', index_col=0)


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.patches import Patch
from matplotlib.gridspec import GridSpec



def plot_corr(df, platform, figsize=(10, 12)):
    df = df.copy()

    # -----------------------------
    # Sorting
    # -----------------------------
    dir_rank = {'Positive': 0, 'Negative': 1}
    df['metab_dir_rank'] = df['metab_moca_dir'].map(dir_rank)
    df['metag_dir_rank'] = df['metag_moca_dir'].map(dir_rank)

    df_sorted_x = df.sort_values(
        by=['Platform', 'metab_dir_rank', 'microbe_metabolite_r'],
        ascending=[True, True, True]
    )
    metabolite_order = df_sorted_x['abbreviated_name'].unique()

    df_sorted_y = df.sort_values(
        by=['metag_dir_rank', 'microbe_metabolite_r'],
        ascending=[True, True]
    )
    microbe_order = df_sorted_y['feature_id'].unique()

    df['metabolite'] = pd.Categorical(df['abbreviated_name'],
                                      categories=metabolite_order, ordered=True)
    df['feature_id'] = pd.Categorical(df['feature_id'],
                                         categories=microbe_order, ordered=True)

    # -----------------------------
    # Cognition values
    # -----------------------------
    metabolite_cog = (
        df.drop_duplicates('metabolite')
          .set_index('metabolite')
          .loc[metabolite_order, 'metab_moca_r']
    )

    microbe_cog = (
        df.drop_duplicates('feature_id')
          .set_index('feature_id')
          .loc[microbe_order, 'metag_moca_r']
    )

    vmax = np.nanmax(np.abs(np.concatenate([
        metabolite_cog.values.astype(float),
        microbe_cog.values.astype(float)
    ])))
    if np.isnan(vmax) or vmax == 0:
        vmax = 1

    # -----------------------------
    # Layout (BOTTOM STRIP)
    # -----------------------------
    fig = plt.figure(figsize=figsize)
    gs = GridSpec(
        nrows=2, ncols=2,
        width_ratios=[0.2, 11],
        height_ratios=[11, 0.4],   # bottom strip instead of top
        wspace=0.02, hspace=0.05
    )

    ax_left = fig.add_subplot(gs[0, 0])   # microbe strip
    ax = fig.add_subplot(gs[0, 1])        # main plot
    ax_bottom = fig.add_subplot(gs[1, 1]) # metabolite strip

    # -----------------------------
    # Left heatmap (microbes)
    # -----------------------------
    sns.heatmap(
        np.array(microbe_cog.values).reshape(-1, 1),
        ax=ax_left,
        cmap='coolwarm',
        center=0,
        vmin=-vmax,
        vmax=vmax,
        cbar=False,
        xticklabels=False,
        yticklabels=False
    )

    # -----------------------------
    # Bottom heatmap (metabolites)
    # -----------------------------
    sns.heatmap(
        np.array([metabolite_cog.values]),
        ax=ax_bottom,
        cmap='coolwarm',
        center=0,
        vmin=-vmax,
        vmax=vmax,
        cbar=False,
        xticklabels=False,
        yticklabels=False
    )

    # -----------------------------
    # Main scatter
    # -----------------------------
    scatter = sns.scatterplot(
        data=df,
        x='metabolite',
        y='feature_id',
        size=np.abs(df['microbe_metabolite_r']),
        hue='microbe_metabolite_r',
        palette='vlag',
        sizes=(40, 300),
        edgecolor='black',
        linewidth=0.3,
        ax=ax
    )

    ax.margins(x=0)
    ax.margins(y=0.1)

    # -----------------------------
    # Platform bands
    # -----------------------------
    xpos = {m: i for i, m in enumerate(metabolite_order)}
    for plat, g in df_sorted_x.groupby('Platform', sort=False):
        pos = [xpos[m] for m in g['abbreviated_name'].unique()]
        start, end = min(pos), max(pos)

        ax.axvspan(
            start - 0.5, end + 0.5,
            color=platform_palette.get(plat, '#999999'),
            alpha=0.06, zorder=0
        )

        # Add platform label centered above each band
        mid = (start + end) / 2
        ax.text(
            mid,
            1.02,                  # a little above the axis
            plat,
            transform=ax.get_xaxis_transform(),
            ha='center',
            va='bottom',
            fontsize=13,
            color=platform_palette.get(plat, '#333333'),
            fontweight='bold',
            clip_on=False
        )
    # -----------------------------
    # Formatting
    # -----------------------------
    ax.tick_params(axis='x', pad=25)   # move x tick labels DOWN
    ax.tick_params(axis='y', pad=25)   # move y tick labels LEFT
    ax.set_xlabel("", fontsize=16)
    ax.set_ylabel("", fontsize=16)
    ax.set_xticklabels(ax.get_xticklabels(), rotation=90, fontsize=14)
    ax.tick_params(axis='y', labelsize=14)
    ax.grid(True, linestyle='--', linewidth=0.3)

    ax_left.axis('off')
    ax_bottom.axis('off')

    # -----------------------------
    # Legends
    # -----------------------------
    handles, labels = ax.get_legend_handles_labels()

    main_legend = ax.legend(
        handles, labels,
        bbox_to_anchor=(1.14, 1),
        title='Correlation (r)',
        loc='upper left'
    )

    plat_handles = [
        Patch(color=c, label=p)
        for p, c in platform_palette.items()
        if p in df['Platform'].unique()
    ]

    if plat_handles:
        ax.add_artist(main_legend)
        ax.legend(
            handles=plat_handles,
            title='Platform',
            bbox_to_anchor=(1.14, 0.45),
            loc='upper left'
        )

    # cognition colorbar
    sm = plt.cm.ScalarMappable(cmap='coolwarm',
                              norm=plt.Normalize(vmin=-vmax, vmax=vmax))
    sm.set_array([])
    # Create a dedicated axis on the LEFT for the colorbar
    cax = fig.add_axes([.86, 0.25, 0.02, 0.6])  
    # [left, bottom, width, height]

    cbar = fig.colorbar(sm, cax=cax)
    cbar.set_label('Association with cognition (r)', fontsize=14)

    # cbar = fig.colorbar(sm, ax=[ax_left, ax_bottom], fraction=0.03, pad=0)
    # cbar.set_label('Association with cognition (r)')

    plt.subplots_adjust(left=0.22, right=0.85, bottom=0.2)
    plt.savefig(f'../figures/{platform}_heatmap_with_bottom_bar.png',
                bbox_inches='tight', dpi=300)
    plt.show()


In [ ]:
df_merged = df_merged.sort_values(by='microbe_metabolite_r')


In [ ]:
df_plot = df_plot.sort_values(by='microbe_metabolite_r')


In [ ]:
df_merged['abbreviated_name']  = df_plot['abbreviated_name']


In [ ]:
platform_palette = {
    # 'Baker': '#E41A1C',
    'UCSD': '#377EB8',
    'Metabolon': '#4DAF4A',
    'Nightingale': '#984EA3',
}

plot_corr(df_merged, 'Metabolite', figsize=(17, 7))


In [ ]:
df_plot_merged[df_plot_merged['abbreviated_name'] == '4-Ace-catechol SO5']


In [ ]:
df_plot_merged['abbreviated_name'].value_counts()


In [ ]:
df_merged.to_csv('microbe_metabolite_results.csv')


## Calculate log-ratios and test with diagnostic groups and diet


In [ ]:
# Define OGUs negative and positivley associated with MOCA
bottom_otus = ['G000436455', 'G000438215', 'G000159435', 'G001042675']

top_otus = ['G900539185', 'G000438095', 'G900547795',
       'G001916165', 'G000155855', 'G900539325', 'G000433515',
       'G000518765', 'G000712055', 'G000421005', 'G003477605', 'G900103835', 'G001603945', 'G000434055', 'G000179635', 'G900095865']


In [ ]:
# Sum top and bottom microbes
top_sum = non_clr[top_otus].sum(axis=1)
bottom_sum = non_clr[bottom_otus].sum(axis=1)


In [ ]:
# Compute log ratio
md[f'moca_log_ratios'] = np.log(top_sum/bottom_sum)


In [ ]:
def plot_groups(df, test_col, group_col, title, ylabel, palette, output, order=None, y=.05):

    jitter_value = 0.1  # Jitter for the stripplot
    
    ft_test = df.dropna(subset=[test_col, group_col])
    groups = ft_test[group_col].unique()
    data = [ft_test[ft_test[group_col] == group][test_col] for group in groups]
    
    if len(groups) == 2:
        kw_statistic, kw_pvalue = mannwhitneyu(*data)
        test_name = 'U'
    else:
        kw_statistic, kw_pvalue = kruskal(*data)
        test_name = 'KW'

    # Plotting
    fig, ax = plt.subplots(figsize=(5, 4))
    sns.boxplot(x=group_col, y=test_col, data=ft_test, width=0.5, order=order, palette=palette)
    sns.stripplot(x=group_col, y=test_col, data=ft_test, color='k', size=5, jitter=jitter_value, alpha=0.5, edgecolor='gray', linewidth=0.5, ax=ax, order=order)

    plt.title(title, fontsize=14)
    p_value_text = f"< .001" if kw_pvalue < 0.001 else f"= {kw_pvalue:.2f}".lstrip('0') if kw_pvalue < 0.01 else f"= {kw_pvalue:.2f}".lstrip('0')
    plt.text(0.5, y, f'{test_name}: {kw_statistic:.1f}, \n $\mathit{{P}}$ value: {kw_pvalue:.2e}', 
             ha='center', va='top', fontsize=11, transform=ax.transAxes)

    plt.xlabel("", fontsize=12)
    plt.ylabel(ylabel, fontsize=12)

    # Setting the linewidth and color of the axes spines
    for spine in ax.spines.values():
        spine.set_linewidth(1)
        spine.set_edgecolor('black')

    plt.tight_layout()
    plt.savefig(output, dpi=600)
    plt.show()


In [ ]:
md = md.replace(np.inf, np.nan)


In [ ]:
palette = {'Cognitively Unimpaired': '#ff7f0e', 'Cognitively Impaired': '#1f77b4'}

plot_groups(md, 'moca_log_ratios', 'Diagnosis', '', "log(+MOCA Taxa/-MOCA Taxa)", palette, '../figures/diagnosis_log_ratio.png', order=None, y=.11)


In [ ]:
males = md[md['SEX'] == 1]


In [ ]:
females = md[md['SEX'] == 2]


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import spearmanr
import seaborn as sns

def plot_corr(df, sex, x, y, group=None): 
    print(x)
    df[x] = df[x].replace({'unknown' : np.nan})
    df = df.copy().dropna(subset=[x, y])
    df[x] = df[x].astype(float)
    # Perform Pearson correlation
    correlation, p_value = spearmanr(df[x], df[y])
    
    if p_value < .05:
        # Plot the scatter plot with a line of best fit for each HIV group
        plt.figure(figsize=(7, 3))
        if group is not None:
            sns.lmplot(x=x, y=y, hue=group, data=df, scatter_kws={'s': 30}, height=5,legend=False)
        else:
            sns.lmplot(x=x, y=y, data=df, scatter_kws={'s': 30}, height=4, legend=False)

        # Add text to show Pearson correlation coefficient on the plot
        plt.text(0.05, 0.05, f'Spearman r = {correlation:.2f}, \nP-value: {p_value:.2e}', ha='left', va='center', 
                 fontsize=12, transform=plt.gca().transAxes)

        # Customize the appearance of the spines (add box including top and right)
        ax = plt.gca()
        # Make the top and right spines visible
        ax.spines['top'].set_visible(True)
        ax.spines['right'].set_visible(True)

        # Label the axes and set a title
        plt.xlabel("log(+MoCA Taxa/-MoCA Taxa)", size=14)
        plt.ylabel('MoCA', size=14)
        plt.title(f'', size=14)

#         # Adjust legend title and location
#         plt.legend(title=f'{group}', loc='best')  # Set title for the legend

        # Adjust layout and display the plot
        plt.tight_layout()
        plt.savefig(f'../figures/{sex}_{x}_{y}_{group}_correlation.png', dpi = 300, bbox_inches='tight', pad_inches=0.1)
        plt.show()
    
    else:
        print(f"{x} not significant: Spearman correlation: {correlation:.2f}, P-value: {p_value:.3f}")


In [ ]:
plot_corr(md, 'All', 'moca_log_ratios', 'MOCA', group=None)


In [ ]:
plot_corr(males, 'Males', 'moca_log_ratios', 'MOCA', group=None)


In [ ]:
plot_corr(females, 'Females', 'moca_log_ratios', 'MOCA', group=None)


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import spearmanr
import seaborn as sns

def plot_corr(df, sex, x, y, group=None): 
    print(x)
    df[x] = df[x].replace({'unknown' : np.nan})
    df = df.copy().dropna(subset=[x, y])
    df[x] = df[x].astype(float)
    # Perform Pearson correlation
    correlation, p_value = spearmanr(df[x], df[y])
    
    if p_value < .05:
        # Plot the scatter plot with a line of best fit for each HIV group
        plt.figure(figsize=(7, 3))
        if group is not None:
            sns.lmplot(x=x, y=y, hue=group, data=df, scatter_kws={'s': 30}, height=5,legend=False)
        else:
            sns.lmplot(x=x, y=y, data=df, scatter_kws={'s': 30}, height=4, legend=False)

        # Add text to show Pearson correlation coefficient on the plot
        plt.text(0.05, .93, f'Spearman r = {correlation:.2f}, \nP-value: {p_value:.2e}', ha='left', va='center', 
                 fontsize=12, transform=plt.gca().transAxes)

        # Customize the appearance of the spines (add box including top and right)
        ax = plt.gca()
        # Make the top and right spines visible
        ax.spines['top'].set_visible(True)
        ax.spines['right'].set_visible(True)

        # Label the axes and set a title
        plt.xlabel("log(+MoCA Taxa/-MoCA Taxa)", size=14)
        plt.ylabel(y, size=14)
        plt.title(f'', size=14)

#         # Adjust legend title and location
#         plt.legend(title=f'{group}', loc='best')  # Set title for the legend

        # Adjust layout and display the plot
        plt.tight_layout()
        plt.savefig(f'../figures/{sex}_{x}_{y}_{group}_correlation.png', dpi = 300, bbox_inches='tight', pad_inches=0.1)
        plt.show()
    
    else:
        print(f"{x} not significant: Spearman correlation: {correlation:.2f}, P-value: {p_value:.3f}")


In [ ]:
plot_corr(md, 'CRAFTDRE', 'moca_log_ratios', 'CRAFTDRE', group=None)


In [ ]:
plot_corr(md, 'UDSBENTD', 'moca_log_ratios', 'UDSBENTD', group=None)


In [ ]:
hei = md.filter(like='HEI')


In [ ]:
food = pd.read_csv('../data/foodomics_adrc_fecal_set_one.csv', dtype={'sample_name': str}).set_index('sample_name')


In [ ]:
food_clr = multiplicative_clr(food.clip(lower=0))


In [ ]:
diet = pd.merge(hei, food_clr, on='sample_name', how='outer')


In [ ]:
diet_resid = residualize(diet, covars)


In [ ]:
diet_resid['moca_log_ratios'] = md['moca_log_ratios']


In [ ]:
diet_resid['metabolite_pc1'] = pd.read_csv('../output_files/sig_metabolites_joint_rpca.csv', dtype={'Unnamed: 0': str}).set_index('Unnamed: 0')['PC1']

diet_resid['metabolite_pc1'] = diet_resid['metabolite_pc1']*-1


In [ ]:
# Collect results for all HEI columns
results = []

for hei in diet_resid.drop(columns=['moca_log_ratios', 'metabolite_pc1']).columns:
    sub = diet_resid[[hei, 'metabolite_pc1', 'moca_log_ratios']].dropna()
    if len(sub) < 3:  # skip tiny n
        continue
    metag_r, metag_p = spearmanr(sub[hei], sub['moca_log_ratios'])
    metab_r, metab_p = spearmanr(sub[hei], sub['metabolite_pc1'])
    results.append({
        'variable': hei,
        'n': len(sub),
        'metag_r': metag_r,
        'metag_p': metag_p,
        'metab_r': metab_r,
        'metab_p': metab_p,
    })
    

diet_df = pd.DataFrame(results)

# One FDR correction across all tests
diet_df['metag_q'] = multipletests(diet_df['metag_p'], method='fdr_bh')[1]
diet_df['metab_q'] = multipletests(diet_df['metab_p'], method='fdr_bh')[1]


In [ ]:
diet_df = diet_df[(diet_df['metag_q'] < .05) | (diet_df['metab_q'] < .05)]


In [ ]:
diet_df = diet_df.set_index('variable')


In [ ]:
diet_resid


In [ ]:
diet_df['summed_r'] = diet_df['metag_r'] + diet_df['metab_r']


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

# Start from your diet_df
df_plot = diet_df.copy()

# Add category
df_plot['category'] = df_plot.index.to_series().apply(
    lambda x: 'HEI2015' if x.startswith('HEI2015') else 'Foodomics'
)

# Reset index so feature names are a column
df_plot = df_plot.reset_index().rename(columns={'variable': 'feature'})

# Sort by combined effect
df_plot = df_plot.sort_values('summed_r')

# Colors for diet category
color_map = {
    'HEI2015': '#2A9D8F',
    'Foodomics': '#E76F51'
}

# Bar positions
y = np.arange(len(df_plot))
bar_h = 0.38

fig, ax = plt.subplots(figsize=(8, 8))

# Draw bars manually
for i, row in df_plot.iterrows():
    pos = df_plot.index.get_loc(i)
    color = color_map[row['category']]
    
    # Microbiome bar
    ax.barh(
        y[pos] - bar_h/2,
        row['metag_r'],
        height=bar_h,
        color=color,
        edgecolor='black',
        linewidth=1
    )
    
    # Metabolome bar
    ax.barh(
        y[pos] + bar_h/2,
        row['metab_r'],
        height=bar_h,
        color=color,
        edgecolor='black',
        linewidth=1,
        hatch='//'
    )

# Axis formatting
ax.set_yticks(y)
ax.set_yticklabels(df_plot['feature'])
ax.axvline(0, color='black', lw=1)
ax.set_xlabel('Spearman r')
ax.set_ylabel('')
ax.set_title('')

# Custom legend
legend_elements = [
    Patch(facecolor='#2A9D8F', edgecolor='black', label='HEI2015'),
    Patch(facecolor='#E76F51', edgecolor='black', label='Foodomics'),
    Patch(facecolor='white', edgecolor='black', label='Microbiome'),
    Patch(facecolor='white', edgecolor='black', hatch='//', label='Metabolome')
]

ax.legend(handles=legend_elements, bbox_to_anchor=(1.02, 1), loc='upper left')

plt.tight_layout()
plt.savefig('../figures/diet_multiomics.png', dpi=300, bbox_inches='tight')
plt.show()


In [ ]:
from scipy.stats import rankdata
md['greens_rank'] = rankdata(md['HEI2015_Greens_Beans'])


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import mannwhitneyu, kruskal

test_col = 'greens_rank'
group_col = 'Diagnosis'

ft_test = md.dropna(subset=[test_col, group_col])
groups = ft_test[group_col].unique()
data = [ft_test[ft_test[group_col] == group][test_col] for group in groups]

if len(groups) == 2:
    kw_statistic, kw_pvalue = mannwhitneyu(*data)
    test_name = 'U'
else:
    kw_statistic, kw_pvalue = kruskal(*data)
    test_name = 'KW'

fig, ax = plt.subplots(figsize=(6, 5))

sns.violinplot(
    data=ft_test,
    x=group_col,
    y=test_col,
    inner=None,
    cut=0,
    linewidth=1,
    ax=ax
)

sns.stripplot(
    data=ft_test,
    x=group_col,
    y=test_col,
    color='black',
    alpha=0.4,
    jitter=0.15,
    ax=ax
)

p_value_text = (
    "< .001" if kw_pvalue < 0.001
    else f"= {kw_pvalue:.3f}".lstrip('0') if kw_pvalue < 0.01
    else f"= {kw_pvalue:.2f}".lstrip('0')
)

ax.text(
    0.5, 0.03,
    f'{test_name}: {kw_statistic:.1f}\n$\\mathit{{P}}$ value {p_value_text}',
    ha='center',
    va='bottom',
    fontsize=11,
    transform=ax.transAxes
)

ax.set_xlabel("")
ax.set_ylabel("HEI2015 Greens & Beans", fontsize=12)

for spine in ax.spines.values():
    spine.set_linewidth(1)
    spine.set_edgecolor('black')

fig.tight_layout()
fig.savefig("../figures/greens_beans_by_diagnosis.png", dpi=600)
plt.show()


In [ ]:
md[md['Diagnosis'] == 'Cognitively Impaired']['Eicosadienoic acid'].mean()


In [ ]:
md[md['Diagnosis'] == 'Cognitively Unimpaired']['Eicosadienoic acid'].mean()


In [ ]:
# plot_groups(md, 'moca_log_ratios', 'Diagnosis', '', "log(+MOCA Taxa/-MOCA Taxa)", palette, '../figures/diagnosis_log_ratio.png', order=None, y=.11)


In [ ]:
diet_resid_filt = diet_resid[df_plot['feature']]


In [ ]:
diet_resid_filt['moca_res'] = cog_resid


In [ ]:
# Collect results for all HEI columns
results = []

for hei in diet_resid_filt.drop(columns=['moca_res']).columns:
    sub = diet_resid_filt[[hei, 'moca_res']].dropna()
    if len(sub) < 3:  # skip tiny n
        continue
    moca_r, moca_p = spearmanr(sub[hei], sub['moca_res'])
    results.append({
        'variable': hei,
        'n': len(sub),
        'moca_r': moca_r,
        'moca_p': moca_p
    })
    

diet_moca_df = pd.DataFrame(results)

# One FDR correction across all tests
diet_moca_df['moca_p'] = multipletests(diet_moca_df['moca_p'], method='fdr_bh')[1]


In [ ]:
# Collect results for all HEI columns
results = []

for hei in diet_resid.drop(columns=['moca_log_ratios','metabolite_pc1']).columns:
    sub = diet_resid[[hei, 'metabolite_pc1']].dropna()
    if len(sub) < 3:  # skip tiny n
        continue
    r, p = spearmanr(sub[hei], sub['metabolite_pc1'])
    results.append({
        'hei': hei,
        'n': len(sub),
        'moca_r': r,
        'moca_p': p
    })

hei_df = pd.DataFrame(results)

# One FDR correction across all tests
hei_df['moca_q'] = multipletests(hei_df['moca_p'], method='fdr_bh')[1]

hei_df = hei_df.set_index('hei')

# Filter significant results
hei_df = hei_df[hei_df['moca_q'] < 0.05]


## Serial Mediation Analysis


In [ ]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

def _fit_paths(df, x, m1, m2, y, covariates=None):
    """
    Fits:
      (1) m1 ~ a1*x + covs
      (2) m2 ~ a2*x + d*m1 + covs
      (3) y  ~ cprime*x + b1*m1 + b2*m2 + covs

    Returns path coefficients and effects.
    """
    covariates = covariates or []
    cov_str = " + " + " + ".join(covariates) if covariates else ""

    # (1) m1 model
    mod1 = smf.ols(f"{m1} ~ {x}{cov_str}", data=df).fit()
    a1 = mod1.params[x]

    # (2) m2 model
    mod2 = smf.ols(f"{m2} ~ {x} + {m1}{cov_str}", data=df).fit()
    a2 = mod2.params[x]
    d  = mod2.params[m1]

    # (3) y model
    mod3 = smf.ols(f"{y} ~ {x} + {m1} + {m2}{cov_str}", data=df).fit()
    cprime = mod3.params[x]
    b1 = mod3.params[m1]
    b2 = mod3.params[m2]

    # Total effect model: y ~ x + covs
    mod_tot = smf.ols(f"{y} ~ {x}{cov_str}", data=df).fit()
    c_total = mod_tot.params[x]

    # Indirect effects
    ind_micro = a1 * b1               # X -> M1 -> Y
    ind_metab = a2 * b2               # X -> M2 -> Y
    ind_chain = a1 * d * b2           # X -> M1 -> M2 -> Y
    total_ind = ind_micro + ind_metab + ind_chain

    out = {
        "a1": a1, "a2": a2, "d": d, "b1": b1, "b2": b2,
        "cprime": cprime, "c_total": c_total,
        "ind_micro": ind_micro, "ind_metab": ind_metab, "ind_chain": ind_chain,
        "total_ind": total_ind,
        "models": {"m1": mod1, "m2": mod2, "y": mod3, "total": mod_tot}
    }
    return out


def serial_mediation_bootstrap(
    df, x, m1, m2, y, covariates=None,
    n_boot=5000, seed=0, dropna=True
):
    """
    Bootstrap CIs for serial mediation effects.
    """
    covariates = covariates or []

    cols = [x, m1, m2, y] + covariates
    d0 = df[cols].copy()
    if dropna:
        d0 = d0.dropna()

    # Point estimate on full data
    point = _fit_paths(d0, x, m1, m2, y, covariates=covariates)

    # Bootstrap
    rng = np.random.default_rng(seed)
    boot = {k: [] for k in ["a1","a2","d","b1","b2","cprime","c_total",
                           "ind_micro","ind_metab","ind_chain","total_ind"]}

    n = len(d0)
    for _ in range(n_boot):
        idx = rng.integers(0, n, size=n)
        db = d0.iloc[idx]
        est = _fit_paths(db, x, m1, m2, y, covariates=covariates)
        for k in boot:
            boot[k].append(est[k])

    # Summarize into dataframe with percentile CIs
    def ci(arr, lo=2.5, hi=97.5):
        return np.percentile(arr, [lo, hi])

    rows = []
    for k, arr in boot.items():
        arr = np.asarray(arr)
        lo, hi = ci(arr)
        rows.append({
            "effect": k,
            "estimate": point[k],
            "ci_2.5": lo,
            "ci_97.5": hi
        })

    res = pd.DataFrame(rows).set_index("effect")

    # Add proportion mediated (using total effect c_total)
    c_total = point["c_total"]
    if c_total != 0:
        res.loc["prop_mediated_total_ind", ["estimate","ci_2.5","ci_97.5"]] = [
            point["total_ind"] / c_total,
            np.percentile(np.asarray(boot["total_ind"]) / np.asarray(boot["c_total"]), 2.5),
            np.percentile(np.asarray(boot["total_ind"]) / np.asarray(boot["c_total"]), 97.5),
        ]

    return point, res


In [ ]:
md['metabolite_pc1'] = pd.read_csv('../output_files/sig_metabolites_joint_rpca.csv', dtype={'Unnamed: 0': str}).set_index('Unnamed: 0')['PC1']


In [ ]:
# Define variables 
X  = "HEI2015_Greens_Beans"
M1 = "metabolite_pc1"
M2 = "moca_log_ratios"
Y  = 'MOCA'

covs = ["NACCAGE", "SEX", "site"]

point, results = serial_mediation_bootstrap(
    df=md, x=X, m1=M1, m2=M2, y=Y,
    covariates=covs,
    n_boot=5000, seed=42
)

print(results.loc[["ind_chain", "ind_micro", "ind_metab", "total_ind", "cprime", "c_total",
                   "prop_mediated_total_ind"]])


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf

# ---------- helpers ----------
def fit_ols(df, formula):
    return smf.ols(formula, data=df).fit()

def partial_residual_xy(df, y, x, covariates):
    """
    Partial residual plot data for effect of x on y adjusting for covariates:
      y_res = residuals of y ~ covs
      x_res = residuals of x ~ covs
    Then plot y_res vs x_res.
    """
    cov_str = " + ".join(covariates)
    m_y = fit_ols(df, f"{y} ~ {cov_str}") if covariates else None
    m_x = fit_ols(df, f"{x} ~ {cov_str}") if covariates else None

    y_res = m_y.resid if m_y is not None else df[y] - df[y].mean()
    x_res = m_x.resid if m_x is not None else df[x] - df[x].mean()
    return x_res, y_res

def add_regline(ax, x, y):
    # simple regression line for plotting
    x = np.asarray(x)
    y = np.asarray(y)
    mask = np.isfinite(x) & np.isfinite(y)
    x = x[mask]; y = y[mask]
    if len(x) < 3:
        return
    b = np.polyfit(x, y, 1)
    xs = np.linspace(x.min(), x.max(), 100)
    ax.plot(xs, b[0]*xs + b[1])

def annotate_ci(ax, text, xy=(0.02, 0.98), fontsize=9):
    ax.text(xy[0], xy[1], text, transform=ax.transAxes,
            ha="left", va="top", fontsize=fontsize)

def plot_serial_mediation_figure(
    df,
    X, M1, M2, Y,
    covs=None,
    effects_table=None,  
    savepath=None
):
    covs = covs or []

    # subset & dropna
    cols = [X, M1, M2, Y] + covs
    d = df[cols].dropna().copy()

    # Fit path models (for plotting + arrow labels)
    cov_str = (" + " + " + ".join(covs)) if covs else ""

    mod_m1 = fit_ols(d, f"{M1} ~ {X}{cov_str}")
    mod_m2 = fit_ols(d, f"{M2} ~ {X} + {M1}{cov_str}")
    mod_y  = fit_ols(d, f"{Y}  ~ {X} + {M1} + {M2}{cov_str}")
    mod_tot = fit_ols(d, f"{Y} ~ {X}{cov_str}")

    a1 = mod_m1.params[X]
    a2 = mod_m2.params[X]
    dd = mod_m2.params[M1]
    b1 = mod_y.params[M1]
    b2 = mod_y.params[M2]
    cprime = mod_y.params[X]
    c_total = mod_tot.params[X]

    # If user provided bootstrap effects table, use it for forest/path annotations
    # Otherwise compute "naive" point effects (no bootstrap CIs).
    if effects_table is None:
        effects_table = pd.DataFrame({
            "estimate": {
                "ind_chain": a1 * dd * b2,
                "ind_micro": a1 * b1,
                "ind_metab": a2 * b2,
                "total_ind": (a1*b1) + (a2*b2) + (a1*dd*b2),
                "cprime": cprime,
                "c_total": c_total,
            },
            "ci_2.5": np.nan,
            "ci_97.5": np.nan
        })

    # ---------- build figure layout ----------
    fig = plt.figure(figsize=(14, 8))
    gs = fig.add_gridspec(2, 3, width_ratios=[1.15, 1, 1], height_ratios=[1, 1], wspace=0.35, hspace=0.35)

    ax_path = fig.add_subplot(gs[:, 0])
    ax1 = fig.add_subplot(gs[0, 1])
    ax2 = fig.add_subplot(gs[0, 2])
    ax3 = fig.add_subplot(gs[1, 1])
    ax_forest = fig.add_subplot(gs[1, 2])

    # ---------- Panel A: path diagram ----------
    ax_path.set_axis_off()

    # node positions (in axes coords)
    pos = {
        "X":  (0.10, 0.70),
        "M1": (0.55, 0.82),
        "M2": (0.55, 0.52),
        "Y":  (0.90, 0.65)
    }

    # draw nodes
    def node(ax, xy, label):
        ax.text(xy[0], xy[1], label, transform=ax.transAxes,
                ha="center", va="center", fontsize=11,
                bbox=dict(boxstyle="round,pad=0.35", fc="white", ec="black"))

    node(ax_path, pos["X"],  f"Diet\n({X})")
    node(ax_path, pos["M1"], f"Microbiome\n({M1})")
    node(ax_path, pos["M2"], f"Metabolites\n({M2})")
    node(ax_path, pos["Y"],  f"Cognition\n({Y})")

    # arrows
    def arrow(ax, p1, p2, text=None, yoff=0.0):
        ax.annotate(
            "", xy=p2, xytext=p1, xycoords=ax.transAxes, textcoords=ax.transAxes,
            arrowprops=dict(arrowstyle="->", lw=1.8)
        )
        if text is not None:
            mx = (p1[0] + p2[0]) / 2
            my = (p1[1] + p2[1]) / 2 + yoff
            ax.text(mx, my, text, transform=ax.transAxes, ha="center", va="center", fontsize=9)

    # arrow labels (path coefficients)
    arrow(ax_path, pos["X"], pos["M1"], text=f"a1 = {a1:.3f}", yoff=0.03)
    arrow(ax_path, pos["X"], pos["M2"], text=f"a2 = {a2:.3f}", yoff=-0.03)
    arrow(ax_path, pos["M1"], pos["M2"], text=f"d = {dd:.3f}", yoff=0.03)
    arrow(ax_path, pos["M1"], pos["Y"], text=f"b1 = {b1:.3f}", yoff=0.03)
    arrow(ax_path, pos["M2"], pos["Y"], text=f"b2 = {b2:.3f}", yoff=-0.03)
    arrow(ax_path, pos["X"], pos["Y"],  text=f"c′ = {cprime:.3f}", yoff=-0.05)

    # annotate key bootstrap effects on the path panel
    def eff_line(name, label):
        if name in effects_table.index:
            est = effects_table.loc[name, "estimate"]
            lo = effects_table.loc[name, "ci_2.5"]
            hi = effects_table.loc[name, "ci_97.5"]
            if np.isfinite(lo) and np.isfinite(hi):
                return f"{label}: {est:.3f} [{lo:.3f}, {hi:.3f}]"
            else:
                return f"{label}: {est:.3f}"

    text_lines = [
        eff_line("ind_chain", "Indirect (chain) a1·d·b2"),
        eff_line("ind_metab", "Indirect (via M2) a2·b2"),
        eff_line("ind_micro", "Indirect (via M1) a1·b1"),
        eff_line("total_ind", "Total indirect"),
        eff_line("c_total", "Total effect (c)"),
    ]
    annotate_ci(ax_path, "Mediation effects (bootstrap CI):\n" + "\n".join([t for t in text_lines if t is not None]),
                xy=(0.02, 0.40), fontsize=9)

    ax_path.set_title("Serial mediation model", fontsize=13, pad=10)

    # ---------- Panel B: Diet -> Microbe (raw) ----------
    ax1.scatter(d[X], d[M1], alpha=0.7)
    add_regline(ax1, d[X], d[M1])
    ax1.set_xlabel(X)
    ax1.set_ylabel(M1)
    ax1.set_title("Diet → Microbiome")
    annotate_ci(ax1, f"a1 = {a1:.3f}\nR² = {mod_m1.rsquared:.2f}")

    # ---------- Panel C: Microbe -> Metab (partial residuals) ----------
    # Effect of M1 on M2 controlling for X + covs
    covs_m2 = [X] + covs  # adjust for diet + covs
    x_res, y_res = partial_residual_xy(d, y=M2, x=M1, covariates=covs_m2)
    ax2.scatter(x_res, y_res, alpha=0.7)
    add_regline(ax2, x_res, y_res)
    ax2.set_xlabel(f"{M1}")
    ax2.set_ylabel(f"{M2}")
    ax2.set_title("Microbiome → Metabolites")
    annotate_ci(ax2, f"d = {dd:.3f}\nModel R² = {mod_m2.rsquared:.2f}")

    # ---------- Panel D: Metab -> Cognition (partial residuals) ----------
    # Effect of M2 on Y controlling for X + M1 + covs
    covs_y = [X, M1] + covs
    x_res2, y_res2 = partial_residual_xy(d, y=Y, x=M2, covariates=covs_y)
    ax3.scatter(x_res2, y_res2, alpha=0.7)
    add_regline(ax3, x_res2, y_res2)
    ax3.set_xlabel(f"{M2}")
    ax3.set_ylabel(f"{Y}")
    ax3.set_title("Metabolites → Cognition")
    annotate_ci(ax3, f"b2 = {b2:.3f}\nModel R² = {mod_y.rsquared:.2f}")

    # ---------- Panel E: Forest plot of effects (use bootstrap CIs if available) ----------
    forest_effects = ["ind_chain", "ind_metab", "ind_micro", "total_ind", "cprime", "c_total"]
    fe = effects_table.loc[[e for e in forest_effects if e in effects_table.index]].copy()

    # order top->bottom
    fe = fe.iloc[::-1]
    y_pos = np.arange(len(fe))
    est = fe["estimate"].values
    lo = fe["ci_2.5"].values
    hi = fe["ci_97.5"].values

    ax_forest.axvline(0, lw=1)
    ax_forest.errorbar(est, y_pos,
                       xerr=[est - lo, hi - est],
                       fmt="o", capsize=3)
    ax_forest.set_yticks(y_pos)
    ax_forest.set_yticklabels(fe.index)
    ax_forest.set_xlabel("Effect (estimate with 95% CI)")
    ax_forest.set_title("Direct & indirect effects")

    # overall title
    fig.suptitle("Diet → Microbiome → Metabolites → Cognition: Serial Mediation", fontsize=14, y=0.98)

    if savepath:
        fig.savefig(savepath, dpi=300, bbox_inches="tight")

    return fig


In [ ]:
fig = plot_serial_mediation_figure(md, X, M1, M2, Y, covs=covs, effects_table=results, savepath="../figures/serial_mediation.png")
plt.show()
